# 🏃‍♀️💨 ADK 2 Orchestration — Your Marathon Coach Adventure! 🏅

Welcome, Agent Architect! 🎉 In this notebook you'll build a **Marathon Race Day Coach** and, one runnable rung at a time, master ADK 2's **three orchestration patterns**.

By the end of this adventure, you'll be able to:

- 🧩 **Compose graph workflows** — mix plain functions and agents as peers, fan out in parallel, and route with a plain `if`-statement.
- 🤝 **Coordinate collaborative agents** — let an LLM pick the *right* specialists for each question and run them in parallel.
- 🌳 **Grow dynamic workflows** — let the LLM shape the work at runtime, with recursion kept safely bounded in code.

| Level | Idea |
|---|---|
| **L0** 🐣 | `Agent` + `Runner` — the atom |
| **L1** 🔗 | first `Workflow`: a function node and an agent node are peers |
| **L2a** 🌤️ | **Pillar 1a** — parallel fetch + `JoinNode` (one agent) |
| **L2b** 🚦 | **Pillar 1b** — add the deterministic router → 1 of 3 agents |
| **L3** 🤝 | **Pillar 2** — coordinator picks a dynamic subset of 6 specialists, in parallel |
| **L4a** 🌱 | **Pillar 3a** — decompose → flat parallel research (runtime width) |
| **L4b** 🌳 | **Pillar 3b** — add recursive spawning (runtime depth) |
| **L5** 🧭 | which pattern, when (reading) |

**The through-line:** known structure → known team / variable subset → unknown shape → choose the right one 🎯

```
    ___       ___       ___       ___       ___
   |o o|     |^_^|     |•‿•|     |^o^|     |=^.^|
   |_-_|     |_-_|     |_-_|     |_-_|     |_-_|
    L0        L1       L2a·b       L3      L4a·b·L5
   ready →   flows →   graph →   team →   dynamic →  🏁
```

> 🏁 Run the cells **top to bottom**. Ready? Let's go! 👇

## Author ✍️

Hi, I'm **Qingyue (Annie) Wang**, a Developer Advocate and AI Engineer at **Google**, passionate about helping developers build with AI and cloud technologies :)

If you have questions about this notebook, reach me on [LinkedIn](https://www.linkedin.com/in/anniewangtech/), [X](https://twitter.com/anniewangtech), or email anniewangtech0510@gmail.com

```
  (\__/)
  (•ㅅ•)
  /づ  🏃   Enjoy building AI Agents — now go run your marathon! :)
```

---
## 🔑 Part 0 · Setup & Authentication

First things first — let's get your gear on! 🎽 This step installs the exact ADK 2 version the tutorial was verified on, then wires up your **Google AI Studio** API key.

1. 👉 Get a free key at **[aistudio.google.com/app/apikey](https://aistudio.google.com/app/apikey)** — click *Create API key*, copy it (starts with `AIza…`).
2. 🔑 In Colab, click the **Secrets** icon (left sidebar) → *Add new secret* → name it **`GOOGLE_API_KEY`**, paste the value, and toggle **Notebook access ON**.
3. Run the two cells below.

In [ ]:
# Pin the exact ADK 2 version this tutorial was verified on.
%pip install -q "google-adk==2.3.0" python-dotenv pydantic nest_asyncio
import nest_asyncio; nest_asyncio.apply()   # let Colab's running loop accept nested awaits
print("\u2713 installed")

In [ ]:
import os

# Google AI Studio API key — add GOOGLE_API_KEY in the 🔑 Secrets panel (or paste when prompted).
try:
    from google.colab import userdata
    key = userdata.get("GOOGLE_API_KEY")
except Exception:
    import getpass
    key = getpass.getpass("Enter your Google AI Studio API key: ")

os.environ["GOOGLE_API_KEY"] = "".join(key.split())   # drop ALL whitespace/newlines (not just the ends)
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "False"     # use AI Studio, not Vertex AI
print("\u2705 API key set \u2014 using Google AI Studio.")

---
## 📦 Shared building blocks

Structured I/O is how ADK 2 moves typed data between function nodes and agents — like passing a clean baton 🏃‍♀️➡️🏃. These Pydantic schemas + canned marathon scenarios are reused from L2 onward. **Run this cell once**, then keep going.

In [ ]:
# Generated from shared/schemas.py + shared/scenarios.py by notebooks/build.py.
# (No `from __future__ import annotations` — deferred string annotations break
#  pydantic forward-refs for nested models in a single notebook namespace.)
MODEL = "gemini-flash-latest"

from typing import Literal

from pydantic import BaseModel, Field


# ─── Pillar 1 (graph workflow) ────────────────────────────────────────────────

class WeatherData(BaseModel):
    temp_f: float
    wind_mph: float
    wind_direction: str
    humidity_pct: int
    conditions: str


class CourseData(BaseModel):
    name: str
    distance_mi: float
    total_elevation_gain_ft: int
    hardest_mile: int
    hardest_mile_grade_pct: float


class FitnessData(BaseModel):
    avg_pace_per_mile_sec: int
    longest_recent_run_mi: float
    weekly_mileage_mi: int


class RaceStrategy(BaseModel):
    target_finish: str = Field(
        description="Clean finish time only, format H:MM:SS. Example: '3:30:00'. "
        "Do NOT include explanations, parentheticals, or extra commentary."
    )
    pacing_advice: str = Field(description="One concise sentence about race pacing strategy.")
    fueling_plan: str = Field(description="One concise sentence about hydration and nutrition.")
    gear: str = Field(description="One concise sentence about clothing and equipment.")
    key_warning: str = Field(description="One concise sentence flagging the biggest risk for this race.")


class BundledRunData(BaseModel):
    """Shape of the JoinNode payload — keys match the upstream function names."""
    fetch_weather: WeatherData
    analyze_course: CourseData
    pull_fitness: FitnessData


# ─── Pillar 2 (collaborative agents) ──────────────────────────────────────────

class SpecialistInput(BaseModel):
    """Payload the concierge coordinator hands to each specialist subagent."""
    user_question: str = Field(description="The runner's free-form question.")
    current_strategy: RaceStrategy = Field(description="The strategy generated in L2.")
    runner_data: BundledRunData = Field(description="The original bundled weather/course/fitness data.")


class SpecialistResponse(BaseModel):
    """Structured response each specialist returns to the coordinator."""
    concern_level: Literal["none", "minor", "moderate", "serious"] = Field(
        description="Severity of the concern this specialist raises."
    )
    recommendation: str = Field(description="One concise sentence with the actionable recommendation.")
    reasoning: str = Field(description="One concise sentence with the why, referencing specific data.")


# ─── Pillar 3 (dynamic workflow) ──────────────────────────────────────────────

class DecomposerOutput(BaseModel):
    """Output of the decompose agent — the research plan."""
    plan_summary: str = Field(
        description="One sentence explaining how you decomposed the question."
    )
    sub_questions: list[str] = Field(
        description="3-7 specific, non-overlapping sub-questions that together "
                    "comprehensively answer the user's main question.",
        min_length=3,
        max_length=7,
    )


class ResearchFinding(BaseModel):
    """Output of one research agent invocation — findings on a single sub-question."""
    summary: str = Field(
        description="Concise 2-3 sentence summary of findings on this sub-question."
    )
    key_facts: list[str] = Field(
        description="3-5 specific factual points or actionable insights discovered.",
        min_length=2,
        max_length=5,
    )
    needs_deeper: bool = Field(
        description="True if these findings reveal a topic that warrants deeper "
                    "recursive investigation. Set False if the question is fully answered."
    )
    deeper_questions: list[str] = Field(
        default_factory=list,
        description="If needs_deeper is True, 1-3 specific, well-formed deeper "
                    "questions to investigate. Empty list if needs_deeper is False.",
        max_length=3,
    )


class DeepResearchBriefing(BaseModel):
    """Final synthesized output of the deep research workflow."""
    headline: str = Field(
        description="One-sentence headline summarizing the most important takeaway."
    )
    sections: list[str] = Field(
        description="3-6 thematic paragraphs covering the research areas.",
        min_length=3,
        max_length=6,
    )
    key_warnings: list[str] = Field(
        description="2-4 actionable warnings or critical considerations.",
        min_length=1,
        max_length=4,
    )
    summary: str = Field(
        description="Closing paragraph that ties everything together actionably."
    )

import os
from typing import Any



SCENARIOS: dict[str, dict[str, Any]] = {
    "HOT": {
        "weather": WeatherData(
            temp_f=78, wind_mph=12, wind_direction="headwind",
            humidity_pct=70, conditions="sunny",
        ),
        "course": CourseData(
            name="Boston Marathon", distance_mi=26.2,
            total_elevation_gain_ft=815, hardest_mile=20,
            hardest_mile_grade_pct=4.5,
        ),
        "fitness": FitnessData(
            avg_pace_per_mile_sec=450,  # 7:30 / mile
            longest_recent_run_mi=22, weekly_mileage_mi=55,
        ),
    },
    "NORMAL": {
        "weather": WeatherData(
            temp_f=58, wind_mph=4, wind_direction="calm",
            humidity_pct=55, conditions="overcast",
        ),
        "course": CourseData(
            name="Berlin Marathon", distance_mi=26.2,
            total_elevation_gain_ft=240, hardest_mile=0,
            hardest_mile_grade_pct=0.5,
        ),
        "fitness": FitnessData(
            avg_pace_per_mile_sec=420,  # 7:00 / mile
            longest_recent_run_mi=24, weekly_mileage_mi=65,
        ),
    },
    "COLD": {
        "weather": WeatherData(
            temp_f=35, wind_mph=15, wind_direction="crosswind",
            humidity_pct=65, conditions="cloudy",
        ),
        "course": CourseData(
            name="Chicago Marathon", distance_mi=26.2,
            total_elevation_gain_ft=140, hardest_mile=0,
            hardest_mile_grade_pct=0.3,
        ),
        "fitness": FitnessData(
            avg_pace_per_mile_sec=440,  # 7:20 / mile
            longest_recent_run_mi=22, weekly_mileage_mi=60,
        ),
    },
}


def scenario() -> dict[str, Any]:
    """Return the currently selected scenario dict (HOT / NORMAL / COLD)."""
    name = os.environ.get("MARATHON_SCENARIO", "HOT").upper()
    if name not in SCENARIOS:
        raise ValueError(f"Unknown scenario {name!r}; expected HOT/NORMAL/COLD")
    return SCENARIOS[name]


def slow_mo() -> float:
    """Multiplier on simulated fetch latencies (env var MARATHON_SLOW_MO)."""
    try:
        return float(os.environ.get("MARATHON_SLOW_MO", "1.0"))
    except ValueError:
        return 1.0

print("\u2713 schemas + scenarios ready")

---
## 🐣 L0 · Your First Agent — the Pace Coach

Every marathon starts with one step. Two objects: an **`Agent`** reasons; a **`Runner`** executes it and streams events. That's the atom — everything after this is just more agents in more interesting shapes.

```
+---------------------------------------------+
|           🏃  pace_coach   (Agent)          |
|---------------------------------------------|
|  model : gemini-flash-latest                |
|  role  : friendly, concise marathon coach   |
|  runs  : Runner  >  Session  >  events      |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys


from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes




MODEL = "gemini-flash-latest"

# 1) Define an agent — a model plus a role. That's it.
pace_coach = Agent(
    name="pace_coach",
    model=MODEL,
    instruction=(
        "You are a friendly, concise marathon coach. Answer the runner's question "
        "in 3-4 sentences. Be specific and practical. No preamble."
    ),
)


def _event_text(event) -> str | None:
    """Pull human-readable text out of an event, whichever field carries it."""
    if getattr(event, "message", None) and getattr(event.message, "parts", None):
        chunks = [p.text for p in event.message.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def ask(question: str) -> None:
    # 2) A Runner executes the agent inside a session and streams back events.
    runner = Runner(
        node=pace_coach,
        app_name="l0_first_agent",
        session_service=InMemorySessionService(),
        auto_create_session=True,
    )

    # 3) Wrap the user's question as a Content message and run.
    message = gtypes.Content(role="user", parts=[gtypes.Part(text=question)])
    print(f"\n🏃 You: {question}\n")
    print("🧠 Coach: ", end="", flush=True)
    async for event in runner.run_async(
        user_id="runner_1",
        session_id="session_1",
        new_message=message,
    ):
        text = _event_text(event)
        if text:
            print(text, end="", flush=True)
    print("\n")


def main() -> None:
    question = " ".join(sys.argv[1:]) or "I'm running my first marathon in 6 weeks. What's the single most important thing to get right?"
    asyncio.run(ask(question))

await ask("What is the most common mistake first-time marathoners make?")

---
## 🔗 L1 · Your First Workflow — code meets model

A plain Python function and an LLM agent are **both just nodes** in one `edges` list. Predictable work stays a function (0 LLM); only reasoning is an agent. `START ──► fetch_conditions (0 LLM) ──► advise (1 LLM)`.

```
+---------------------------------------------+
|          🔗  l1_workflow  (Workflow)        |
|---------------------------------------------|
|  START > fetch_conditions > advise          |
|          (function, 0 LLM)   (agent, 1 LLM) |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys

from pydantic import BaseModel

from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import START
from google.adk.sessions import InMemorySessionService




MODEL = "gemini-flash-latest"


# ─── A tiny schema so the function node and the agent speak the same shape ─────

class Conditions(BaseModel):
    temp_f: float
    wind_mph: float
    conditions: str


# ─── Node 1: a plain function. No model, no cost — just prepares data. ─────────

def fetch_conditions(node_input):
    """In a real app this hits a weather API. Here it returns canned data.

    Returning `Event(output=...)` is the explicit form. A function node may also
    return a bare value or a pydantic model and ADK wraps it for you — the explicit
    form is used here because L2b needs its sibling, `Event(output=..., route=...)`.
    """
    data = Conditions(temp_f=78, wind_mph=12, conditions="sunny")
    print(f"  [fetch_conditions] (function node, 0 LLM) → {data.model_dump()}")
    return Event(output=data.model_dump())


# ─── Node 2: an agent. The only step that calls the model. ────────────────────

advise = Agent(
    name="advise",
    model=MODEL,
    input_schema=Conditions,  # receives the function node's output, validated
    instruction=(
        "You are a marathon coach. Given today's race-day conditions, give the "
        "runner 2-3 sentences of specific advice on pacing and gear for THESE "
        "conditions. Reference the actual temperature and wind."
    ),
)


# ─── The workflow: two nodes, one edge chain. ─────────────────────────────────

workflow = Workflow(
    name="l1_graph_basics",
    description="One function node feeds one agent node.",
    edges=[(START, fetch_conditions, advise)],
)


def _event_text(event) -> str | None:
    if getattr(event, "message", None) and getattr(event.message, "parts", None):
        chunks = [p.text for p in event.message.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def main() -> None:
    runner = Runner(
        node=workflow,
        app_name="l1_graph_basics",
        session_service=InMemorySessionService(),
        auto_create_session=True,
    )
    print("=== L1 workflow: fetch_conditions (function) → advise (agent) ===")
    async for event in runner.run_async(
        user_id="runner_1", session_id="session_1", new_message=None,
    ):
        text = _event_text(event)
        if text:
            print(f"\n🧠 Coach: {text}\n")

await main()

---
## 🌤️ L2a · Parallel Fan-out + JoinNode (Pillar 1a)

Gather race-day data **in parallel** (3 functions, 0 LLM), a **`JoinNode`** bundles it, and one strategy agent writes the plan. No router yet. ✨ Try `run("COLD")` and watch it adapt!

```
+---------------------------------------------+
|       🌤️  l2a_parallel_join  (Workflow)     |
|---------------------------------------------|
|  fetch_weather  --+                          |
|  analyze_course --+--> JoinNode --> strategy |
|  pull_fitness   --+   (parallel,0LLM)   (1)  |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import JoinNode, START
from google.adk.sessions import InMemorySessionService





MODEL = "gemini-flash-latest"


# ─── Fetch nodes: parallel, zero LLM. Simulated latency so you SEE concurrency. ─
#
# Each fetch prints when it starts and finishes, relative to the start of the run.
# That overlap is the actual evidence for the parallel claim — all three start at
# ~0.0s and the fan-out ends when the SLOWEST one does, not when their durations
# add up. (Wall time for the whole run is a bad proxy: it also contains the
# strategy agent's LLM call, which is several seconds on its own.)

_T0 = 0.0


def _stamp(msg: str) -> None:
    print(f"  [t={time.perf_counter() - _T0:4.1f}s] {msg}")


async def _fetch(name: str, seconds: float, key: str):
    _stamp(f"{name} started")
    await asyncio.sleep(seconds * slow_mo())
    _stamp(f"{name} finished")
    return Event(output=scenario()[key].model_dump())


async def fetch_weather(node_input):
    return await _fetch("fetch_weather", 1.5, "weather")


async def analyze_course(node_input):
    return await _fetch("analyze_course", 2.0, "course")


async def pull_fitness(node_input):
    return await _fetch("pull_fitness", 1.0, "fitness")


# ─── Join: bundle the three parallel results into one typed payload. ──────────

join_inputs = JoinNode(name="join_inputs")


# ─── One strategy agent (no routing yet). ─────────────────────────────────────

strategy = Agent(
    name="strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction="""You are a marathon coach. You receive BundledRunData (weather from
fetch_weather, course from analyze_course, fitness from pull_fitness). Produce a
RaceStrategy that ADAPTS to whatever the conditions are — hot, cold, or ideal.
Cite actual numbers (temps, mile numbers, the runner's pace). Each field 1-2
short sentences.""",
)


# ─── The workflow: fan-out, join, one agent. ──────────────────────────────────

root = Workflow(
    name="l2a_parallel_join",
    description="Parallel data gathering + a single strategy agent.",
    edges=[
        (START, fetch_weather, join_inputs),
        (START, analyze_course, join_inputs),
        (START, pull_fitness, join_inputs),
        (join_inputs, strategy),
    ],
)


def _event_text(event):
    msg = getattr(event, "message", None)
    if msg and getattr(msg, "parts", None):
        chunks = [p.text for p in msg.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def run(scenario_name="HOT"):
    global _T0
    os.environ["MARATHON_SCENARIO"] = scenario_name.upper()
    print(f"=== L2a · scenario: {scenario_name.upper()} ===")
    runner = Runner(node=root, app_name="l2a", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = _T0 = time.perf_counter()
    async for event in runner.run_async(user_id="u1", session_id="s1", new_message=None):
        t = _event_text(event)
        if t:
            print(f"\n🏁 RaceStrategy:\n{t}")
    print(f"\n  Total wall time: {time.perf_counter()-t0:.1f}s "
          f"(fan-out + join + 1 LLM call — the fetch timestamps above are the parallel evidence)")

await run("HOT")

---
## 🚦 L2b · Add the Deterministic Router (Pillar 1b)

Now a **deterministic `if`-statement router** branches on temperature to one of three specialized agents — hot / normal / cold. **Net cost: 1 LLM call.** 🔁 Try `run("NORMAL")`.

```
+---------------------------------------------+
|      🚦  marathon_strategy  (Workflow)      |
|---------------------------------------------|
|  3 parallel fetches > JoinNode > router     |
|  > hot / normal / cold      (1 LLM call)    |
+---------------------------------------------+
```

In [ ]:
import asyncio
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import JoinNode, START
from google.adk.sessions import InMemorySessionService





MODEL = "gemini-flash-latest"


# ─── Fetch nodes (same as L2a): parallel, zero LLM. ───────────────────────────

async def fetch_weather(node_input):
    await asyncio.sleep(1.5 * slow_mo())
    return Event(output=scenario()["weather"].model_dump())


async def analyze_course(node_input):
    await asyncio.sleep(2.0 * slow_mo())
    return Event(output=scenario()["course"].model_dump())


async def pull_fitness(node_input):
    await asyncio.sleep(1.0 * slow_mo())
    return Event(output=scenario()["fitness"].model_dump())


join_inputs = JoinNode(name="join_inputs")


# ─── The new part: a deterministic router. An if-statement, not the model. ────

def route_by_weather(node_input):
    """node_input is the JoinNode payload, keyed by upstream function names.
    Emit `route=` to pick exactly one downstream branch."""
    temp = node_input["fetch_weather"]["temp_f"]
    if temp >= 70:
        route = "HOT"
    elif temp <= 40:
        route = "COLD"
    else:
        route = "NORMAL"
    print(f"  [router] temp={temp}°F → route={route}  (an if-statement, 0 LLM)")
    return Event(output=node_input, route=route)


# ─── Three specialized strategy agents. Only one ever runs. ───────────────────

_FMT = """
You receive BundledRunData (weather/course/fitness). Produce a RaceStrategy.
Cite actual numbers (temps, mile numbers, the runner's pace). Each field 1-2
short sentences.
"""

hot_strategy = Agent(
    name="hot_strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction=f"""You are a marathon coach planning a race in HOT conditions.
Heat is the primary risk: slow down, hydrate aggressively, dress cool, set a
goal time SLOWER than ideal.{_FMT}""",
)

normal_strategy = Agent(
    name="normal_strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction=f"""You are a marathon coach planning a race in IDEAL conditions.
This is a PR-attempt day. Recommend even or slightly negative splits; the main
risk is going out too fast on cool, fast-feeling pavement.{_FMT}""",
)

cold_strategy = Agent(
    name="cold_strategy", model=MODEL,
    input_schema=BundledRunData, output_schema=RaceStrategy,
    instruction=f"""You are a marathon coach planning a race in COLD conditions.
Recommend layered, shed-able gear, a careful warm-up, and fueling that accounts
for cold-suppressed thirst. If winds are strong, advise drafting.{_FMT}""",
)


# ─── The workflow: L2a + the router and its dict-edge. ────────────────────────

root = Workflow(
    name="marathon_strategy",
    description="Parallel data gathering + deterministic routing to one of three agents.",
    edges=[
        (START, fetch_weather, join_inputs),
        (START, analyze_course, join_inputs),
        (START, pull_fitness, join_inputs),
        (join_inputs, route_by_weather),
        (route_by_weather, {
            "HOT": hot_strategy,
            "NORMAL": normal_strategy,
            "COLD": cold_strategy,
        }),
    ],
)


def _event_text(event):
    msg = getattr(event, "message", None)
    if msg and getattr(msg, "parts", None):
        chunks = [p.text for p in msg.parts if getattr(p, "text", None)]
        if chunks:
            return "".join(chunks)
    out = getattr(event, "output", None)
    return out if isinstance(out, str) else None


async def run(scenario_name="HOT"):
    os.environ["MARATHON_SCENARIO"] = scenario_name.upper()
    print(f"=== L2b · scenario: {scenario_name.upper()} ===")
    runner = Runner(node=root, app_name="l2b", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = time.perf_counter()
    async for event in runner.run_async(user_id="u1", session_id="s1", new_message=None):
        t = _event_text(event)
        if t:
            print(f"\n🏁 RaceStrategy:\n{t}")
    print(f"\n  Wall time: {time.perf_counter()-t0:.1f}s (3 fetches parallel; strategy = 1 LLM call)")

await run("HOT")

---
## 🤝 L3 · Collaborative Agents — the Race Concierge (Pillar 2)

Known **team** (6 specialists 🩺🌦️⏱️🎽🥤🧠); the **question** picks who answers. The coordinator emits several delegation-tool calls in one turn; because the specialists are `mode="single_turn"`, ADK runs the chosen subset **in parallel** (each in an isolated branch), then synthesizes one reply. 💡 Change the question and re-run — the contrast *is* the lesson!

**The 3 collaboration modes** — this is the first level that sets `mode` at all, because earlier levels' agents are *workflow nodes*, which already default to `single_turn`. Subagents default to `chat`, so here the argument does real work. Set it on **subagents only**, never the coordinator: `chat` = free multi-turn with the user (the subagent default); `task` = asks a clarifying question, then auto-returns; `single_turn` = no user interaction, auto-returns, **runs in parallel**. This demo uses `single_turn` — the only one that runs concurrently, which is the whole point here.

*Note: occasionally a specialist returns prose not JSON and you'll see a validation warning — the coordinator recovers. 🙂*

```
+-----------------------------------------------+
|       🤝  race_concierge  (coordinator)       |
|-----------------------------------------------|
|  6 specialists (single_turn):                 |
|  medical.weather.pacing.gear.nutrition.mental |
|  reads the question > picks a subset >        |
|  runs them in parallel > synthesizes 1 answer |
+-----------------------------------------------+
```

In [ ]:
import asyncio
import json
import os
import sys
import time


from google.adk import Agent, Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes





MODEL = "gemini-flash-latest"


# ─── Specialist factory: six narrow single_turn agents. ───────────────────────

def _specialist(name: str, domain: str, focus: str) -> Agent:
    return Agent(
        name=name, model=MODEL, mode="single_turn",
        # `description` is NOT optional decoration: ADK turns each subagent into a
        # tool and uses this text as that tool's description, so it is what the
        # coordinator actually reads when choosing the subset. Skip it and you are
        # asking the coordinator to route on the agent's *name* alone.
        description=f"Marathon {domain} specialist. Consult for: {focus}.",
        input_schema=SpecialistInput, output_schema=SpecialistResponse,
        instruction=f"""You are a marathon {domain} specialist. Answer ONLY questions
within your domain. Focus: {focus}

Given the question, the current race strategy, and the runner's data, produce a
SpecialistResponse:
- concern_level: "none" | "minor" | "moderate" | "serious"
- recommendation: ONE concrete actionable sentence
- reasoning: ONE sentence citing specific numbers from the strategy or runner data

If the question is outside your domain, set concern_level="none" and say so briefly.""",
    )


medical_specialist = _specialist("medical_specialist", "medical",
    "injury risk, pain, when to stop, hydration safety, heat stroke")
weather_specialist = _specialist("weather_specialist", "weather",
    "heat, cold, wind, rain, race-day forecast adjustments")
pacing_specialist = _specialist("pacing_specialist", "pacing",
    "pace strategy, mile splits, heart rate, target finish time")
gear_specialist = _specialist("gear_specialist", "gear",
    "clothing, shoes, accessories (hats, glasses, gloves), drop-bag contents")
nutrition_specialist = _specialist("nutrition_specialist", "nutrition",
    "fueling plan, gels, electrolytes, hydration timing, pre-race meals")
mental_specialist = _specialist("mental_specialist", "mental",
    "race mindset, motivation, pre-race anxiety, mid-race low points")


# ─── Coordinator: holds the 6 as sub_agents and picks the subset. ─────────────

race_concierge = Agent(
    name="race_concierge",
    model=MODEL,
    sub_agents=[
        medical_specialist, weather_specialist, pacing_specialist,
        gear_specialist, nutrition_specialist, mental_specialist,
    ],
    instruction="""You are a marathon race day concierge. The runner already has a
strategy and is asking a follow-up. Six specialists are available:
medical, weather, pacing, gear, nutrition, mental.

For each question:
1. DECIDE which specialists are genuinely relevant — be precise. Examples:
   "My knee hurts" → medical only. "What about fueling?" → nutrition only.
   "Should I race today?" → medical + weather + pacing. "Anything I should worry
   about?" → all 6. Do NOT invoke specialists whose domain doesn't apply.
2. Call the relevant specialist tools IN PARALLEL (multiple function calls in one
   turn). Each takes a SpecialistInput: user_question (verbatim), current_strategy
   and runner_data (forward both from the user's initial message).
3. SYNTHESIZE their responses into one answer, under 4 sentences, leading with the
   most important concern.

The user's message contains the strategy + runner data as JSON — extract and forward them.""",
)


# ─── A run harness that shows WHICH specialists fired. ─────────────────────────

def _build_message(question: str) -> gtypes.Content:
    """Give the coordinator a question plus a (canned) strategy + runner data."""
    s = scenario()
    runner_data = BundledRunData(
        fetch_weather=s["weather"], analyze_course=s["course"], pull_fitness=s["fitness"],
    )
    # Derive the strategy from the SAME scenario the runner data came from.
    # (Hardcoding a heat-stroke warning here would contradict the 35°F numbers a
    # learner sees after `MARATHON_SCENARIO=COLD` — the specialists would be
    # handed a self-contradicting brief.)
    w = s["weather"]
    if w.temp_f >= 70:
        pacing = "Start 20s/mile slower than goal pace to bank against the heat."
        gear = "Light singlet, cap, sunglasses."
        warning = (f"{w.temp_f:.0f}°F + {w.humidity_pct}% humidity — "
                   "real heat-stroke risk at your usual 7:30 pace.")
    elif w.temp_f <= 40:
        pacing = "Hold goal pace; the cold masks effort, so do not start too fast."
        gear = "Long sleeves, gloves, throwaway layer for the corral."
        warning = (f"{w.temp_f:.0f}°F — hypothermia risk if you slow down late; "
                   "keep a dry layer at the finish.")
    else:
        pacing = "Even splits — conditions are close to ideal for goal pace."
        gear = "Singlet and shorts; no weather adjustment needed."
        warning = (f"{w.temp_f:.0f}°F and {w.conditions} — no weather red flags; "
                   "the risk is going out too fast.")
    strategy = RaceStrategy(
        target_finish="3:32:00",
        pacing_advice=pacing,
        fueling_plan="Electrolytes every aid station.",
        gear=gear,
        key_warning=warning,
    )
    body = (
        f"{question}\n\n"
        f"--- context ---\n"
        f"current_strategy: {strategy.model_dump_json()}\n"
        f"runner_data: {runner_data.model_dump_json()}"
    )
    return gtypes.Content(role="user", parts=[gtypes.Part(text=body)])


async def ask(question: str) -> None:
    runner = Runner(
        node=race_concierge, app_name="l3_concierge",
        session_service=InMemorySessionService(), auto_create_session=True,
    )
    print(f"\n💬 Question: {question}\n")
    specialist_names = {
        "medical_specialist", "weather_specialist", "pacing_specialist",
        "gear_specialist", "nutrition_specialist", "mental_specialist",
    }
    dispatched: list[str] = []
    returned: list[str] = []
    final_text = ""
    t0 = time.perf_counter()
    async for event in runner.run_async(
        user_id="runner_1", session_id="s1", new_message=_build_message(question),
    ):
        # Function calls to specialists reveal the chosen subset. (ADK's built-in
        # transfer_to_agent tool may also appear — we only count our specialists.)
        msg = getattr(event, "message", None)
        for part in getattr(msg, "parts", None) or []:
            fc = getattr(part, "function_call", None)
            if fc and fc.name in specialist_names and fc.name not in dispatched:
                dispatched.append(fc.name)
                print(f"  [t={time.perf_counter()-t0:4.1f}s] DISPATCH → {fc.name}")
            # Each specialist's answer coming back. Printing these timestamps is the
            # evidence for the parallel claim: every DISPATCH shares one timestamp
            # (one turn, many calls) and the REPLIES all land inside one short
            # window — not spaced out end-to-end the way serial calls would be.
            fr = getattr(part, "function_response", None)
            if fr and fr.name in specialist_names and fr.name not in returned:
                returned.append(fr.name)
                print(f"  [t={time.perf_counter()-t0:4.1f}s]   ↩ {fr.name} replied")
            # Only the COORDINATOR's text is the final answer. Specialists stream
            # their raw JSON through this same event stream, so without this author
            # check the last specialist's JSON blob can be printed as the answer.
            if getattr(part, "text", None) and getattr(event, "author", None) == "race_concierge":
                final_text = part.text
    print(f"\n  Specialists chosen: {dispatched or ['(none)']}")
    print(f"  Total time: {time.perf_counter()-t0:.1f}s")
    if final_text:
        print(f"\n🧠 Concierge:\n{final_text}\n")


def main() -> None:
    question = " ".join(sys.argv[1:]) or "Should I race today?"
    asyncio.run(ask(question))

await ask("Should I race today?")

---
## 🌱 L4a · Runtime-Sized Fan-out — Deep Research (Pillar 3a)

An open-ended question is **decomposed** into N sub-questions (**width chosen at runtime**), each researched **in parallel**, then synthesized. One level deep — no recursion yet. Note `retry_config=` on the worker: a parallel worker cancels all siblings when one child fails, so without a retry a single transient error throws away the whole run.

⚠️ 5-9 live LLM calls (1 decompose + 3-7 research + 1 synthesize), ~20-30s.

```
+---------------------------------------------+
|      🌱  l4a_flat_research  (Workflow)       |
|---------------------------------------------|
|  decompose > research x N (parallel) >      |
|  synthesize     (width chosen at runtime)   |
+---------------------------------------------+
```

In [ ]:
import asyncio
import json
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import RetryConfig, START, node
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes





MODEL = "gemini-flash-latest"


# ─── Three single-turn agents: decompose, research, synthesize. ───────────────

decompose_agent = Agent(
    name="decompose_agent", model=MODEL,
    output_schema=DecomposerOutput,
    instruction="""You are a research coordinator for marathon/endurance questions.
Break the user's open-ended question into 3-7 specific, non-overlapping,
independently-researchable sub-questions, each covering a distinct angle.""",
)

research_agent = Agent(
    name="research_agent", model=MODEL,
    output_schema=ResearchFinding,
    instruction="""You are a marathon research specialist. Given ONE specific
research question, produce a finding: a 2-3 sentence summary and 3-5 specific
insights. (For this level, keep needs_deeper=False.)""",
)

synthesize_agent = Agent(
    name="synthesize_agent", model=MODEL,
    output_schema=DeepResearchBriefing,
    instruction="""You are a marathon coach synthesizing a JSON list of findings
into one briefing for the runner: a HEADLINE, 3-6 thematic SECTIONS with specific
facts, 2-4 KEY_WARNINGS, and a closing SUMMARY. Write for the runner, be direct.""",
)


def _coerce(payload, schema_cls):
    if isinstance(payload, schema_cls):
        return payload
    if isinstance(payload, dict):
        return schema_cls.model_validate(payload)
    if isinstance(payload, str):
        return schema_cls.model_validate_json(payload)
    if hasattr(payload, "parts") and payload.parts:
        text = getattr(payload.parts[0], "text", None)
        if text:
            return schema_cls.model_validate_json(text)
    raise ValueError(f"Cannot coerce {type(payload).__name__} into {schema_cls.__name__}")


def _extract_text(node_input):
    if isinstance(node_input, str):
        return node_input
    if hasattr(node_input, "parts") and node_input.parts:
        return getattr(node_input.parts[0], "text", None) or str(node_input)
    return str(node_input)


# ─── Node 1: decompose into a runtime-sized list of sub-questions. ────────────

@node(rerun_on_resume=True)
async def decompose(ctx, node_input):
    user_query = _extract_text(node_input)
    plan = _coerce(await ctx.run_node(decompose_agent, node_input=user_query), DecomposerOutput)
    print(f"  [decompose] {len(plan.sub_questions)} sub-questions (width chosen at runtime):")
    for q in plan.sub_questions:
        print(f"    • {q[:80]}")
    yield Event(output=[{"question": q, "original_query": user_query} for q in plan.sub_questions])


# ─── Node 2: parallel_worker — one research task per item. Flat (no recursion). ─

# ─── Bounding the RATE, not just the shape. ───────────────────────────────────
#
# The fan-out width is already bounded by the schema (max_length=7). This bounds
# how it FAILS: a parallel worker cancels every sibling and re-raises the moment
# one child raises, so without a retry a single transient 429 throws away a whole
# run — including every call you already paid for. `retry_config` is applied to
# the INNER per-item node, so each branch retries on its own and a transient blip
# is absorbed before it can take the others down.
#
# Note `max_concurrency` is a real field on the parallel worker but is NOT
# reachable through the public `node()` API in ADK 2.3.0 — so on a rate-limited
# key, the schema bound is what keeps the in-flight count sane.
RESEARCH_RETRY = RetryConfig(max_attempts=3, initial_delay=2.0, backoff_factor=2.0)

@node(parallel_worker=True, rerun_on_resume=True,
      retry_config=RESEARCH_RETRY)
async def research_topic(ctx, node_input):
    question = node_input["question"]
    ctxq = node_input.get("original_query", "")
    print(f"  [research] {question[:70]}")
    finding = _coerce(await ctx.run_node(
        research_agent,
        node_input=f"ORIGINAL QUERY: {ctxq}\n\nRESEARCH QUESTION: {question}",
    ), ResearchFinding)
    yield Event(output={"question": question, "summary": finding.summary, "key_facts": finding.key_facts})


# ─── Node 3: synthesize the flat list of findings. ────────────────────────────

@node(rerun_on_resume=True)
async def synthesize(ctx, node_input):
    print(f"  [synthesize] merging {len(node_input)} findings")
    briefing = _coerce(await ctx.run_node(
        synthesize_agent, node_input=json.dumps(node_input, indent=2)), DeepResearchBriefing)
    yield Event(output={"briefing": briefing.model_dump(), "findings": node_input})


l4a_workflow = Workflow(
    name="l4a_flat_research",
    description="Decompose (runtime width) → flat parallel research → synthesize.",
    edges=[(START, decompose, research_topic, synthesize)],
)


async def run(query="Tell me everything I should know about racing the Boston Marathon."):
    print(f"=== L4a · flat research ===\n  QUERY: {query}\n")
    runner = Runner(node=l4a_workflow, app_name="l4a", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = time.perf_counter()
    final = None
    async for event in runner.run_async(
        user_id="u1", session_id="s1",
        new_message=gtypes.Content(role="user", parts=[gtypes.Part(text=query)]),
    ):
        out = getattr(event, "output", None)
        if isinstance(out, dict) and "briefing" in out:
            final = out
    if final:
        b = final["briefing"]
        print(f"\n  {len(final['findings'])} sub-questions researched in parallel | {time.perf_counter()-t0:.1f}s")
        print(f"\n📋 {b['headline']}\n")
        for i, s in enumerate(b["sections"], 1):
            print(f"  {i}. {s[:180]}")

await run()

---
## 🌳 L4b · Add Recursive Spawning (Pillar 3b)

Now each finding can **recursively spawn** deeper questions via `ctx.run_node` (**depth chosen at runtime**), safely bounded by `MAX_DEPTH`. Recursion runs **inside the framework** — you keep tracing & checkpointing. 🌲

⚠️ 5-30 live LLM calls, 20-45s — ceiling = 1 + 7 + 7×3 + 1.

```
+---------------------------------------------+
|        🌳  deep_research  (Workflow)         |
|---------------------------------------------|
|  decompose > research > (recurse: children) |
|  > synthesize    MAX_DEPTH = 2  guard-rail  |
+---------------------------------------------+
```

In [ ]:
import asyncio
import json
import os
import sys
import time


from google.adk import Agent, Event, Runner, Workflow
from google.adk.workflow import RetryConfig, START, node
from google.adk.sessions import InMemorySessionService
from google.genai import types as gtypes





MODEL = "gemini-flash-latest"

# The boundary kept in CODE. The LLM decides width and depth, but never past this.
MAX_DEPTH = 2


decompose_agent = Agent(
    name="decompose_agent", model=MODEL,
    output_schema=DecomposerOutput,
    instruction="""You are a research coordinator for marathon/endurance questions.
Break the user's open-ended question into 3-7 specific, non-overlapping,
independently-researchable sub-questions, each covering a distinct angle.""",
)

research_agent = Agent(
    name="research_agent", model=MODEL,
    output_schema=ResearchFinding,
    instruction="""You are a marathon research specialist. Given ONE specific
research question, produce a 2-3 sentence summary and 3-5 insights. Set
needs_deeper=True ONLY if your findings surface a genuinely narrow technical
sub-topic that deserves its own investigation; then give 1-3 specific
deeper_questions.""",
)

synthesize_agent = Agent(
    name="synthesize_agent", model=MODEL,
    output_schema=DeepResearchBriefing,
    instruction="""You are a marathon coach synthesizing a nested JSON tree of
findings into one briefing: a HEADLINE, 3-6 SECTIONS combining related findings
with specific facts, 2-4 KEY_WARNINGS, and a closing SUMMARY. Write for the runner.""",
)


def _coerce(payload, schema_cls):
    if isinstance(payload, schema_cls):
        return payload
    if isinstance(payload, dict):
        return schema_cls.model_validate(payload)
    if isinstance(payload, str):
        return schema_cls.model_validate_json(payload)
    if hasattr(payload, "parts") and payload.parts:
        text = getattr(payload.parts[0], "text", None)
        if text:
            return schema_cls.model_validate_json(text)
    raise ValueError(f"Cannot coerce {type(payload).__name__} into {schema_cls.__name__}")


def _extract_text(node_input):
    if isinstance(node_input, str):
        return node_input
    if hasattr(node_input, "parts") and node_input.parts:
        return getattr(node_input.parts[0], "text", None) or str(node_input)
    return str(node_input)


@node(rerun_on_resume=True)
async def decompose(ctx, node_input):
    user_query = _extract_text(node_input)
    plan = _coerce(await ctx.run_node(decompose_agent, node_input=user_query), DecomposerOutput)
    print(f"  [decompose] {len(plan.sub_questions)} sub-questions (width chosen at runtime):")
    for q in plan.sub_questions:
        print(f"    • {q[:80]}")
    yield Event(output=[{"question": q, "depth": 1, "original_query": user_query} for q in plan.sub_questions])


# ─── Bounding the RATE, not just the shape. ───────────────────────────────────
#
# The fan-out width is already bounded by the schema (max_length=7). This bounds
# how it FAILS: a parallel worker cancels every sibling and re-raises the moment
# one child raises, so without a retry a single transient 429 throws away a whole
# run — including every call you already paid for. `retry_config` is applied to
# the INNER per-item node, so each branch retries on its own and a transient blip
# is absorbed before it can take the others down.
#
# Note `max_concurrency` is a real field on the parallel worker but is NOT
# reachable through the public `node()` API in ADK 2.3.0 — so on a rate-limited
# key, the schema bound is what keeps the in-flight count sane.
RESEARCH_RETRY = RetryConfig(max_attempts=3, initial_delay=2.0, backoff_factor=2.0)

@node(parallel_worker=True, rerun_on_resume=True,
      retry_config=RESEARCH_RETRY)
async def research_topic(ctx, node_input):
    question = node_input["question"]
    depth = node_input["depth"]
    ctxq = node_input.get("original_query", "")
    print(f"  [research d={depth}] {question[:66]}")
    finding = _coerce(await ctx.run_node(
        research_agent,
        node_input=f"ORIGINAL QUERY: {ctxq}\n\nRESEARCH QUESTION: {question}",
    ), ResearchFinding)

    children = []
    # The boundary in code: only spawn if the LLM asked AND we're under MAX_DEPTH.
    if finding.needs_deeper and finding.deeper_questions and depth < MAX_DEPTH:
        deeper = [{"question": dq, "depth": depth + 1, "original_query": ctxq}
                  for dq in finding.deeper_questions]
        print(f"  [research d={depth}] spawning {len(deeper)} deeper (depth chosen at runtime)")
        children = await ctx.run_node(research_topic, node_input=deeper)   # recursive fan-out

    yield Event(output={"question": question, "depth": depth, "summary": finding.summary,
                        "key_facts": finding.key_facts, "children": children})


@node(rerun_on_resume=True)
async def synthesize(ctx, node_input):
    print(f"  [synthesize] merging {len(node_input)} top-level findings + their children")
    briefing = _coerce(await ctx.run_node(
        synthesize_agent, node_input=json.dumps(node_input, indent=2)), DeepResearchBriefing)
    yield Event(output={"briefing": briefing.model_dump(), "research_tree": node_input})


l4b_workflow = Workflow(
    name="deep_research",
    description="Decompose → recursive parallel research → synthesize.",
    edges=[(START, decompose, research_topic, synthesize)],
)


async def run(query="Tell me everything I should know about racing the Boston Marathon."):
    print(f"=== L4b · recursive deep research ===\n  QUERY: {query}\n")
    runner = Runner(node=l4b_workflow, app_name="l4b", session_service=InMemorySessionService(), auto_create_session=True)
    t0 = time.perf_counter()
    final = None
    async for event in runner.run_async(
        user_id="u1", session_id="s1",
        new_message=gtypes.Content(role="user", parts=[gtypes.Part(text=query)]),
    ):
        out = getattr(event, "output", None)
        if isinstance(out, dict) and "briefing" in out:
            final = out
    if final:
        tree = final["research_tree"]
        top = len(tree)
        deeper = sum(len(n.get("children", [])) for n in tree)
        b = final["briefing"]
        print(f"\n  Tree (runtime-decided): {top} top-level + {deeper} recursive children | {time.perf_counter()-t0:.1f}s")
        print(f"\n📋 {b['headline']}\n")
        for i, s in enumerate(b["sections"], 1):
            print(f"  {i}. {s[:180]}")

await run()

---
## 🧭 L5 · Which Pattern, When? (the finish line 🏁)

One question picks your pattern:

```
Can you draw the workflow before the input arrives?
├─ YES ───────────────────────────────► Pillar 1 · graph workflow   (L2a/L2b) 🚦
└─ NO
   ├─ Known team, request picks the subset? ─► Pillar 2 · collaborative (L3) 🤝
   └─ Does the shape depend on the input?  ──► Pillar 3 · dynamic       (L4a/L4b) 🌳
```

**Not** "2.0 can do what 1.x couldn't" — 1.x could build all of it. 2.0 gives each shape a **more direct home**, so known control flow leaves the prompt and becomes structure you can see and test. And they **compose**: a graph node can call a collaborative coordinator; a specialist can launch a dynamic workflow. 🧩

> 🏅 *Predictable work stays functions; clear rules become explicit routing; reasoning uses the model.*
> 🌟 *Let the LLM shape the work, but keep the boundaries in code.*
> 🎯 *Match the pattern to the shape of your problem.*

```
  \o/   You finished the marathon! 🏁🎉
   |    You now know all three ADK 2 orchestration patterns.
  / \   Go build something amazing — and keep the boundaries in code. 💪
```

🐾 *Made with love by Annie — happy building!*